Fuel Efficiency Prediction — Clean Backend



In [ ]:
!pip install lime --break-system-packages

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 8.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=1b9242281eca4c6940efb7e8754ee10a7fbe9fbc69176eea5fcf596bd071ac92
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


In [ ]:
import sys
import platform
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import xgboost as xgb
import sklearn
import shap
from lime.lime_tabular import LimeTabularExplainer
from scipy.stats import spearmanr

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold, StratifiedKFold
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error






In [ ]:
RANDOM_STATE = 42

# CURRENT_YEAR is the reference year used to compute "vehicle/car age"
# features (car_age = CURRENT_YEAR - model_year / vehicle_age = CURRENT_YEAR - Year).
# It is fixed at 2026 because that is the year this analysis/revision was
# conducted. STATE THIS EXPLICITLY IN THE METHODS SECTION, e.g.:
#   "Vehicle age was computed as 2026 minus the model year, where 2026
#    denotes the reference year of this analysis."
# This single sentence pre-empts the "why 2026?" reviewer question.
CURRENT_YEAR = 2026

N_BOOTSTRAP  = 1000
rng          = np.random.default_rng(RANDOM_STATE)
kf           = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

XGB_PARAM_GRID = {
    "n_estimators":  [200, 300],
    "max_depth":     [4, 6],
    "learning_rate": [0.05, 0.1],
    "subsample":     [0.8, 1.0],
}

RF_PARAM_GRID = {
    "n_estimators":      [200, 300],
    "max_depth":         [8, 10, 12],
    "min_samples_split": [2, 5],
}


In [ ]:

# ============================================================
# SHARED HELPER FUNCTIONS
# ============================================================

def report_metrics(name, y_true, y_pred):
    r2   = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    print(f"{name:26s} | R2: {r2:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f}")
    return r2, rmse, mae


def bootstrap_r2_ci(y_true, y_pred, n_bootstrap=N_BOOTSTRAP):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    n = len(y_true)
    scores = np.empty(n_bootstrap)
    for i in range(n_bootstrap):
        idx = rng.integers(0, n, n)
        scores[i] = r2_score(y_true[idx], y_pred[idx])
    lo, hi = np.percentile(scores, [2.5, 97.5])
    return lo, hi


def stratified_quantile_cv(model, X, y, n_splits=5, n_bins=5, random_state=RANDOM_STATE):
    """
    Plain KFold can produce highly variable folds when the target is
    heterogeneous (Car Specs spans economy hatchbacks to supercars, so a
    random fold can end up dominated by one segment). This bins the
    target into quantiles and uses StratifiedKFold on those bins, so
    every fold sees a similar mix of low/medium/high-MPG vehicles --
    giving a more reliable (lower-variance) CV estimate of generalization.
    """
    y_arr = np.array(y)
    X_arr = np.array(X)
    y_bins = pd.qcut(y_arr, q=n_bins, labels=False, duplicates="drop")
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    scores = []
    for train_idx, test_idx in skf.split(X_arr, y_bins):
        m = clone(model)
        m.fit(X_arr[train_idx], y_arr[train_idx])
        preds = m.predict(X_arr[test_idx])
        scores.append(r2_score(y_arr[test_idx], preds))
    return np.array(scores)


In [ ]:

def tune_xgb(X_train_sc, y_train):
    grid = GridSearchCV(
        xgb.XGBRegressor(random_state=RANDOM_STATE, verbosity=0),
        XGB_PARAM_GRID, cv=5, scoring="r2", n_jobs=-1
    )
    grid.fit(X_train_sc, y_train)
    return grid.best_estimator_, grid.best_params_


def tune_rf(X_train_sc, y_train):
    grid = GridSearchCV(
        RandomForestRegressor(random_state=RANDOM_STATE),
        RF_PARAM_GRID, cv=5, scoring="r2", n_jobs=-1
    )
    grid.fit(X_train_sc, y_train)
    return grid.best_estimator_, grid.best_params_


In [ ]:

def plot_actual_vs_predicted(y_true, y_pred, title, filename):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]

    plt.figure(figsize=(6, 6))
    plt.scatter(y_true, y_pred, alpha=0.6, edgecolor="k")
    plt.plot(lims, lims, "r--", label="Ideal (y = x)")
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename, dpi=200)
    plt.close()





In [ ]:
def plot_residuals(y_true, y_pred, title_prefix, filename_scatter, filename_hist):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    residuals = y_true - y_pred

    plt.figure(figsize=(6, 5))
    plt.scatter(y_pred, residuals, alpha=0.6, edgecolor="k")
    plt.axhline(0, color="r", linestyle="--")
    plt.xlabel("Predicted")
    plt.ylabel("Residual (Actual - Predicted)")
    plt.title(f"{title_prefix} — Residual Scatter")
    plt.tight_layout()
    plt.savefig(filename_scatter, dpi=200)
    plt.close()

    plt.figure(figsize=(6, 5))
    plt.hist(residuals, bins=25, edgecolor="k", alpha=0.7)
    plt.xlabel("Residual")
    plt.ylabel("Frequency")
    plt.title(f"{title_prefix} — Residual Distribution")
    plt.tight_layout()
    plt.savefig(filename_hist, dpi=200)
    plt.close()

    return residuals



In [ ]:
def plot_tree_importance_comparison(rf_model, xgb_model, feature_names, title, filename):
    rf_imp  = rf_model.feature_importances_
    xgb_imp = xgb_model.feature_importances_
    rf_imp  = rf_imp / rf_imp.sum()
    xgb_imp = xgb_imp / xgb_imp.sum()

    x = np.arange(len(feature_names))
    width = 0.35
    plt.figure(figsize=(9, 5))
    plt.bar(x - width / 2, rf_imp, width, label="Random Forest")
    plt.bar(x + width / 2, xgb_imp, width, label="XGBoost")
    plt.xticks(x, feature_names, rotation=45, ha="right")
    plt.ylabel("Normalized Feature Importance")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename, dpi=200)
    plt.close()





In [ ]:
# ============================================================
# PART A: DATASET 1 — Auto MPG (Vehicle Configuration Model)
# ============================================================
print("=" * 60)
print("PART A: Auto MPG Dataset — Configuration-Based Model")
print("=" * 60)

df = pd.read_csv("auto-mpg.csv")
df["horsepower"] = df["horsepower"].replace("?", None).astype(float)
df = df.dropna()
df = df.drop(columns=["car name"], errors="ignore")

# Feature engineering (all derived from Auto MPG columns only)
df["power_to_weight"] = df["horsepower"] / df["weight"]
df["car_age"]         = CURRENT_YEAR - df["model-year"]
df["hp_per_cylinder"] = df["horsepower"] / df["cylinders"]

# FIX: target 'mpg' must NEVER appear in the feature list
#
# FIX (collinearity): 'model-year' is dropped. 'car_age' = CURRENT_YEAR -
# model-year is a perfect linear transform of it, so keeping both is
# redundant duplicate information -- this is exactly why 'car_age' showed
# ~0 SHAP/tree importance earlier (the model was splitting the same
# signal between the two). We keep 'car_age' (the engineered, more
# interpretable version) and drop the raw 'model-year' column.
config_features = [
    "cylinders", "displacement", "horsepower",
    "weight",
    "power_to_weight", "car_age", "hp_per_cylinder"
]

X_config = df[config_features]
y_config = df["mpg"]

X_c_train, X_c_test, y_c_train, y_c_test = train_test_split(
    X_config, y_config, test_size=0.2, random_state=RANDOM_STATE
)

scaler_config = StandardScaler()
X_c_train_sc  = scaler_config.fit_transform(X_c_train)
X_c_test_sc   = scaler_config.transform(X_c_test)

# --- Tune BOTH models ---
model_config, xgb_config_params = tune_xgb(X_c_train_sc, y_c_train)
rf_config, rf_config_params     = tune_rf(X_c_train_sc, y_c_train)
print("Best Params (Config XGBoost):", xgb_config_params)
print("Best Params (Config Random Forest):", rf_config_params)

# --- 5-Fold CV for both ---
cv_r2_xgb_c = cross_val_score(model_config, X_c_train_sc, y_c_train, cv=kf, scoring="r2")
cv_r2_rf_c  = cross_val_score(rf_config,    X_c_train_sc, y_c_train, cv=kf, scoring="r2")
print(f"Config XGBoost 5-Fold CV R2:      {cv_r2_xgb_c.mean():.4f} +/- {cv_r2_xgb_c.std():.4f}")
print(f"Config RandomForest 5-Fold CV R2: {cv_r2_rf_c.mean():.4f} +/- {cv_r2_rf_c.std():.4f}")

# --- Test metrics for both ---
preds_c_xgb = model_config.predict(X_c_test_sc)
preds_c_rf  = rf_config.predict(X_c_test_sc)
r2_c_xgb, rmse_c_xgb, mae_c_xgb = report_metrics("Config XGBoost (test)", y_c_test, preds_c_xgb)
r2_c_rf,  rmse_c_rf,  mae_c_rf  = report_metrics("Config RandomForest (test)", y_c_test, preds_c_rf)

# --- Bootstrap CI (primary model = XGBoost) ---
ci_lo_c, ci_hi_c = bootstrap_r2_ci(y_c_test, preds_c_xgb)
print(f"Config XGBoost Bootstrap 95% CI for R2: [{ci_lo_c:.4f}, {ci_hi_c:.4f}]")

# --- Actual vs Predicted + Residual plots (primary model) ---
plot_actual_vs_predicted(
    y_c_test, preds_c_xgb,
    "Actual vs Predicted — Config Model (XGBoost)",
    "actual_vs_predicted_config.png"
)
plot_residuals(
    y_c_test, preds_c_xgb, "Config Model (XGBoost)",
    "residual_scatter_config.png", "residual_distribution_config.png"
)

# --- RF vs XGBoost native tree-importance comparison ---
plot_tree_importance_comparison(
    rf_config, model_config, config_features,
    "Tree Feature Importance: RF vs XGBoost — Config Model",
    "tree_importance_comparison_config.png"
)

joblib.dump(model_config,    "model_config.pkl")
joblib.dump(rf_config,       "model_config_rf.pkl")
joblib.dump(scaler_config,   "scaler_config.pkl")
joblib.dump(config_features, "features_config.pkl")
print("Model 1 (Config, XGBoost primary + RF benchmark) saved.\n")


PART A: Auto MPG Dataset — Configuration-Based Model
Best Params (Config XGBoost): {'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 200, 'subsample': 0.8}
Best Params (Config Random Forest): {'max_depth': 12, 'min_samples_split': 5, 'n_estimators': 300}
Config XGBoost 5-Fold CV R2:      0.8573 +/- 0.0362
Config RandomForest 5-Fold CV R2: 0.8629 +/- 0.0264
Config XGBoost (test)      | R2: 0.8400 | RMSE: 2.8809 | MAE: 1.9802
Config RandomForest (test) | R2: 0.8640 | RMSE: 2.6561 | MAE: 1.8107
Config XGBoost Bootstrap 95% CI for R2: [0.7296, 0.9149]
Model 1 (Config, XGBoost primary + RF benchmark) saved.



In [ ]:

# ============================================================
# PART B: DATASET 2 — Car Specifications (Brand + Model)
# ============================================================
print("=" * 60)
print("PART B: Car Specifications Dataset — Brand-Based Model")
print("=" * 60)

from google.colab import files
uploaded = files.upload()   # select archive.zip

import zipfile
with zipfile.ZipFile("archive.zip", "r") as zip_ref:
    zip_ref.extractall("car_dataset")

df2 = pd.read_csv("car_dataset/data.csv")
df2.columns = df2.columns.str.strip()

df2 = df2.dropna(subset=["highway MPG", "city mpg", "Engine HP", "Engine Cylinders", "Year"])
df2 = df2[df2["Engine HP"] > 0]
df2 = df2[df2["Engine Cylinders"] > 0]

# avg_mpg is computed ONLY here, from Car-Specs highway/city MPG.
# This is Model 2's target — it is NOT derived from, and never mixed
# with, the Auto MPG dataset used in Part A.
df2["avg_mpg"] = (df2["highway MPG"] * 0.55) + (df2["city mpg"] * 0.45)


def estimate_weight(hp, cyl):
    base = 2500 + (hp - 100) * 8 + (cyl - 4) * 200
    return max(1800, min(6000, base))


df2["weight_est"]       = df2.apply(lambda r: estimate_weight(r["Engine HP"], r["Engine Cylinders"]), axis=1)
df2["vehicle_age"]      = CURRENT_YEAR - df2["Year"]
df2["power_to_weight"]  = df2["Engine HP"] / df2["weight_est"]
df2["hp_per_cylinder"]  = df2["Engine HP"] / df2["Engine Cylinders"]
df2["displacement_est"] = df2["Engine Cylinders"] * 400

df2 = df2.replace([np.inf, -np.inf], np.nan)
df2 = df2.dropna(subset=[
    "Engine HP", "Engine Cylinders", "weight_est", "displacement_est",
    "vehicle_age", "power_to_weight", "hp_per_cylinder", "avg_mpg"
])

# FIX: MSRP removed from the predictive feature set entirely.
# Make / Model / MSRP stay in df2 for dashboard lookup only (see
# predict_by_brand below) — they are never passed into the model.
#
# FIX (collinearity): 'displacement_est' is dropped from the feature
# list. It was computed as Engine Cylinders * 400 -- a fixed multiple of
# a feature already in the model, so it carries zero additional
# information. This is exactly why it showed ~0 SHAP/tree importance
# above. displacement_est is still computed on df2 below (kept as a
# descriptive/derived column in case it's referenced elsewhere), it is
# simply excluded from the model's input features.
brand_features = [
    "Engine HP", "Engine Cylinders", "weight_est",
    "vehicle_age",
    "power_to_weight", "hp_per_cylinder"
]

X_brand = df2[brand_features].copy()
y_brand = df2["avg_mpg"]

X_b_train, X_b_test, y_b_train, y_b_test = train_test_split(
    X_brand, y_brand, test_size=0.2, random_state=RANDOM_STATE
)

scaler_brand = StandardScaler()
X_b_train_sc = scaler_brand.fit_transform(X_b_train)
X_b_test_sc  = scaler_brand.transform(X_b_test)

# --- Tune BOTH sub-models ---
xgb_brand, xgb_brand_params = tune_xgb(X_b_train_sc, y_b_train)
rf_brand,  rf_brand_params  = tune_rf(X_b_train_sc, y_b_train)
print("Best Params (Brand XGBoost):", xgb_brand_params)
print("Best Params (Brand Random Forest):", rf_brand_params)

# --- 5-Fold CV for both sub-models ---
cv_r2_xgb = cross_val_score(xgb_brand, X_b_train_sc, y_b_train, cv=kf, scoring="r2")
cv_r2_rf  = cross_val_score(rf_brand,  X_b_train_sc, y_b_train, cv=kf, scoring="r2")
print(f"Brand XGBoost 5-Fold CV R2:      {cv_r2_xgb.mean():.4f} +/- {cv_r2_xgb.std():.4f}")
print(f"Brand RandomForest 5-Fold CV R2: {cv_r2_rf.mean():.4f} +/- {cv_r2_rf.std():.4f}")

# --- Stratified (quantile-binned) CV: more reliable estimate for a
# heterogeneous target like avg_mpg (economy cars vs. supercars can land
# unevenly across plain random folds, inflating the std of plain KFold) ---
cv_r2_xgb_strat = stratified_quantile_cv(xgb_brand, X_b_train_sc, y_b_train)
cv_r2_rf_strat  = stratified_quantile_cv(rf_brand,  X_b_train_sc, y_b_train)
print(f"Brand XGBoost Stratified CV R2:      {cv_r2_xgb_strat.mean():.4f} +/- {cv_r2_xgb_strat.std():.4f}")
print(f"Brand RandomForest Stratified CV R2: {cv_r2_rf_strat.mean():.4f} +/- {cv_r2_rf_strat.std():.4f}")

# --- Weighted ensemble (as in original design) ---
xgb_b_preds = xgb_brand.predict(X_b_test_sc)
rf_b_preds  = rf_brand.predict(X_b_test_sc)

XGB_W, RF_W = 0.60, 0.40
ens_preds = (XGB_W * xgb_b_preds) + (RF_W * rf_b_preds)

xgb_r2, xgb_rmse, xgb_mae = report_metrics("Brand XGBoost", y_b_test, xgb_b_preds)
rf_r2,  rf_rmse,  rf_mae  = report_metrics("Brand RandomForest", y_b_test, rf_b_preds)
ens_r2, ens_rmse, ens_mae = report_metrics("Brand Ensemble (60/40)", y_b_test, ens_preds)

ci_lo_b, ci_hi_b = bootstrap_r2_ci(y_b_test, ens_preds)
print(f"Brand Ensemble Bootstrap 95% CI for R2: [{ci_lo_b:.4f}, {ci_hi_b:.4f}]")

# --- Actual vs Predicted + Residual plots (final ensemble) ---
plot_actual_vs_predicted(
    y_b_test, ens_preds,
    "Actual vs Predicted — Brand Model (Ensemble)",
    "actual_vs_predicted_brand.png"
)
plot_residuals(
    y_b_test, ens_preds, "Brand Model (Ensemble)",
    "residual_scatter_brand.png", "residual_distribution_brand.png"
)

# --- RF vs XGBoost native tree-importance comparison ---
plot_tree_importance_comparison(
    rf_brand, xgb_brand, brand_features,
    "Tree Feature Importance: RF vs XGBoost — Brand Model",
    "tree_importance_comparison_brand.png"
)

joblib.dump(xgb_brand,      "model_xgb_brand.pkl")
joblib.dump(rf_brand,       "model_rf_brand.pkl")
joblib.dump(scaler_brand,   "scaler_brand.pkl")
joblib.dump(brand_features, "features_brand.pkl")

brand_metrics = {
    "xgb_r2": round(xgb_r2, 4), "xgb_rmse": round(xgb_rmse, 4), "xgb_mae": round(xgb_mae, 4),
    "rf_r2":  round(rf_r2, 4),  "rf_rmse":  round(rf_rmse, 4),  "rf_mae":  round(rf_mae, 4),
    "ens_r2": round(ens_r2, 4), "ens_rmse": round(ens_rmse, 4), "ens_mae": round(ens_mae, 4),
    "xgb_weight": XGB_W, "rf_weight": RF_W,
}
joblib.dump(brand_metrics, "brand_metrics.pkl")
print("Model 2 (Brand: XGBoost + RF + Ensemble) saved.\n")

PART B: Car Specifications Dataset — Brand-Based Model


Saving archive.zip to archive.zip
Best Params (Brand XGBoost): {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 300, 'subsample': 1.0}
Best Params (Brand Random Forest): {'max_depth': 12, 'min_samples_split': 2, 'n_estimators': 200}
Brand XGBoost 5-Fold CV R2:      0.8232 +/- 0.1118
Brand RandomForest 5-Fold CV R2: 0.8202 +/- 0.1119
Brand XGBoost Stratified CV R2:      0.8230 +/- 0.1071
Brand RandomForest Stratified CV R2: 0.8159 +/- 0.1059
Brand XGBoost              | R2: 0.8848 | RMSE: 2.0207 | MAE: 1.2862
Brand RandomForest         | R2: 0.8850 | RMSE: 2.0196 | MAE: 1.2710
Brand Ensemble (60/40)     | R2: 0.8876 | RMSE: 1.9964 | MAE: 1.2707
Brand Ensemble Bootstrap 95% CI for R2: [0.8691, 0.9038]
Model 2 (Brand: XGBoost + RF + Ensemble) saved.



In [ ]:

# ============================================================
# PART C: SHAP EXPLAINABILITY
# (Paper 1 reports ONLY this section)
# ============================================================
print("=" * 60)
print("PART C: SHAP Explainability")
print("=" * 60)

explainer_config   = shap.TreeExplainer(model_config)
X_c_test_df        = pd.DataFrame(X_c_test_sc, columns=config_features)
shap_values_config  = explainer_config(X_c_test_df)

plt.figure()
shap.summary_plot(shap_values_config, X_c_test_df, show=False)
plt.title("SHAP Summary — Config Model (Auto MPG)")
plt.tight_layout()
plt.savefig("shap_summary_config.png", dpi=200)
plt.close()

plt.figure()
shap.summary_plot(shap_values_config, X_c_test_df, plot_type="bar", show=False)
plt.title("SHAP Feature Importance (Bar) — Config Model")
plt.tight_layout()
plt.savefig("shap_bar_config.png", dpi=200)
plt.close()

plt.figure()
shap.plots.waterfall(shap_values_config[0], show=False)
plt.tight_layout()
plt.savefig("shap_local_config_instance0.png", dpi=200)
plt.close()

# Explain the XGBoost sub-model (primary contributor, 60% ensemble weight)
explainer_brand    = shap.TreeExplainer(xgb_brand)
X_b_test_df        = pd.DataFrame(X_b_test_sc, columns=brand_features)
shap_values_brand   = explainer_brand(X_b_test_df)

plt.figure()
shap.summary_plot(shap_values_brand, X_b_test_df, show=False)
plt.title("SHAP Summary — Brand Model (Car Specifications)")
plt.tight_layout()
plt.savefig("shap_summary_brand.png", dpi=200)
plt.close()

plt.figure()
shap.summary_plot(shap_values_brand, X_b_test_df, plot_type="bar", show=False)
plt.title("SHAP Feature Importance (Bar) — Brand Model")
plt.tight_layout()
plt.savefig("shap_bar_brand.png", dpi=200)
plt.close()

plt.figure()
shap.plots.waterfall(shap_values_brand[0], show=False)
plt.tight_layout()
plt.savefig("shap_local_brand_instance0.png", dpi=200)
plt.close()

print("SHAP plots saved: shap_summary_config.png, shap_bar_config.png, "
      "shap_local_config_instance0.png, shap_summary_brand.png, "
      "shap_bar_brand.png, shap_local_brand_instance0.png\n")


PART C: SHAP Explainability
SHAP plots saved: shap_summary_config.png, shap_bar_config.png, shap_local_config_instance0.png, shap_summary_brand.png, shap_bar_brand.png, shap_local_brand_instance0.png



In [ ]:


# ============================================================
# PART C2: LIME EXPLAINABILITY
# (Project backend / Paper 2 only — Paper 1 does not report this)
# ============================================================
print("=" * 60)
print("PART C2: LIME Explainability")
print("=" * 60)

lime_explainer_config = LimeTabularExplainer(
    training_data=X_c_train_sc, feature_names=config_features,
    mode="regression", discretize_continuous=True, random_state=RANDOM_STATE,
)
lime_exp_config_0 = lime_explainer_config.explain_instance(
    X_c_test_sc[0], model_config.predict, num_features=len(config_features)
)
lime_exp_config_0.save_to_file("lime_local_config_instance0.html")
fig = lime_exp_config_0.as_pyplot_figure()
fig.suptitle("LIME Local Explanation — Config Model (instance 0)")
fig.tight_layout()
fig.savefig("lime_local_config_instance0.png", dpi=200)
plt.close(fig)

# Explaining the XGBoost sub-model (same model SHAP explained above) so
# the SHAP vs LIME comparison in Part C3 is apples-to-apples.
lime_explainer_brand = LimeTabularExplainer(
    training_data=X_b_train_sc, feature_names=brand_features,
    mode="regression", discretize_continuous=True, random_state=RANDOM_STATE,
)
lime_exp_brand_0 = lime_explainer_brand.explain_instance(
    X_b_test_sc[0], xgb_brand.predict, num_features=len(brand_features)
)
lime_exp_brand_0.save_to_file("lime_local_brand_instance0.html")
fig = lime_exp_brand_0.as_pyplot_figure()
fig.suptitle("LIME Local Explanation — Brand Model (instance 0)")
fig.tight_layout()
fig.savefig("lime_local_brand_instance0.png", dpi=200)
plt.close(fig)

print("LIME local explanations saved (html + png) for both models.\n")


def lime_global_importance(explainer, predict_fn, X_test_sc, feature_names, n_samples=50):
    """Aggregate |LIME weight| across instances -> approximate global importance."""
    n = min(n_samples, len(X_test_sc))
    importance_sum = {f: 0.0 for f in feature_names}
    for i in range(n):
        exp = explainer.explain_instance(X_test_sc[i], predict_fn, num_features=len(feature_names))
        for feat_desc, weight in exp.as_list():
            for f in feature_names:
                if f in feat_desc:
                    importance_sum[f] += abs(weight)
                    break
    return {f: v / n for f, v in importance_sum.items()}


print("Aggregating LIME global importance over 50 test instances (Config Model)...")
lime_importance_config = lime_global_importance(
    lime_explainer_config, model_config.predict, X_c_test_sc, config_features, n_samples=50
)
print("Aggregating LIME global importance over 50 test instances (Brand Model)...")
lime_importance_brand = lime_global_importance(
    lime_explainer_brand, xgb_brand.predict, X_b_test_sc, brand_features, n_samples=50
)

PART C2: LIME Explainability
LIME local explanations saved (html + png) for both models.

Aggregating LIME global importance over 50 test instances (Config Model)...
Aggregating LIME global importance over 50 test instances (Brand Model)...


In [ ]:


# ============================================================
# PART C3: SHAP vs LIME COMPARISON (Paper 2's core contribution)
# ============================================================
print("=" * 60)
print("PART C3: SHAP vs LIME Comparison")
print("=" * 60)


def compare_shap_lime(shap_values, feature_names, lime_importance_dict, title, filename):
    shap_importance = np.abs(shap_values.values).mean(axis=0)
    shap_dict = dict(zip(feature_names, shap_importance))

    shap_total = sum(shap_dict.values())
    lime_total = sum(lime_importance_dict.values())
    shap_norm = {f: v / shap_total for f, v in shap_dict.items()}
    lime_norm = {f: v / lime_total for f, v in lime_importance_dict.items()}

    rho, pval = spearmanr(
        [shap_norm[f] for f in feature_names],
        [lime_norm[f] for f in feature_names],
    )
    print(f"{title} | Spearman rank correlation (SHAP vs LIME): rho={rho:.4f}, p={pval:.4f}")

    x = np.arange(len(feature_names))
    width = 0.35
    plt.figure(figsize=(9, 5))
    plt.bar(x - width / 2, [shap_norm[f] for f in feature_names], width, label="SHAP (normalized)")
    plt.bar(x + width / 2, [lime_norm[f] for f in feature_names], width, label="LIME (normalized)")
    plt.xticks(x, feature_names, rotation=45, ha="right")
    plt.ylabel("Normalized Importance")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename, dpi=200)
    plt.close()

    return rho, pval


rho_config, p_config = compare_shap_lime(
    shap_values_config, config_features, lime_importance_config,
    "SHAP vs LIME — Config Model (Auto MPG)", "shap_lime_comparison_config.png"
)
rho_brand, p_brand = compare_shap_lime(
    shap_values_brand, brand_features, lime_importance_brand,
    "SHAP vs LIME — Brand Model (Car Specifications)", "shap_lime_comparison_brand.png"
)

xai_comparison_results = {
    "config_spearman_rho": round(float(rho_config), 4),
    "config_spearman_p":   round(float(p_config), 4),
    "brand_spearman_rho":  round(float(rho_brand), 4),
    "brand_spearman_p":    round(float(p_brand), 4),
}
joblib.dump(xai_comparison_results, "xai_comparison_results.pkl")
print("SHAP vs LIME comparison saved.\n")




PART C3: SHAP vs LIME Comparison
SHAP vs LIME — Config Model (Auto MPG) | Spearman rank correlation (SHAP vs LIME): rho=0.8571, p=0.0137
SHAP vs LIME — Brand Model (Car Specifications) | Spearman rank correlation (SHAP vs LIME): rho=0.8857, p=0.0188
SHAP vs LIME comparison saved.



In [ ]:

# ============================================================
# PART D: PREDICTION FUNCTIONS
# ============================================================

def predict_by_config(cylinders, displacement, horsepower, weight, model_year):
    power_to_weight = horsepower / weight
    car_age         = CURRENT_YEAR - model_year
    hp_per_cylinder = horsepower / cylinders

    input_data = pd.DataFrame([[
        cylinders, displacement, horsepower,
        weight,
        power_to_weight, car_age, hp_per_cylinder
    ]], columns=config_features)

    input_scaled = scaler_config.transform(input_data)
    mpg_pred     = model_config.predict(input_scaled)[0]
    kmpl_pred    = mpg_pred * 0.4251

    print(f"\n{'=' * 35}\n [Config-Based Prediction]\n MPG : {mpg_pred:.2f}\n KMPL: {kmpl_pred:.2f}\n{'=' * 35}")
    return mpg_pred, kmpl_pred


def predict_by_brand(brand_name, model_name):
    match = df2[
        (df2["Make"].str.lower()  == brand_name.lower()) &
        (df2["Model"].str.lower() == model_name.lower())
    ]
    if match.empty:
        print(f"Car not found: {brand_name} {model_name}")
        return None

    row = match.iloc[0]
    input_data   = pd.DataFrame([[row[c] for c in brand_features]], columns=brand_features)
    input_scaled = scaler_brand.transform(input_data)

    xgb_pred = xgb_brand.predict(input_scaled)[0]
    rf_pred  = rf_brand.predict(input_scaled)[0]
    ens_pred = (XGB_W * xgb_pred) + (RF_W * rf_pred)
    kmpl_pred = ens_pred * 0.4251

    # MSRP shown here ONLY as dashboard lookup metadata — never a model input
    msrp = row["MSRP"] if "MSRP" in df2.columns else None

    msg = (f"\n{'=' * 35}\n [Brand-Based Prediction]\n Car : {brand_name} {model_name}"
           f"\n MPG : {ens_pred:.2f}\n KMPL: {kmpl_pred:.2f}")
    if msrp is not None:
        msg += f"\n MSRP (lookup only, not a model feature): {msrp}"
    msg += f"\n{'=' * 35}"
    print(msg)
    return ens_pred, kmpl_pred


predict_by_config(cylinders=4, displacement=1500, horsepower=120, weight=2800, model_year=2020)
predict_by_brand("BMW", "1 Series")





 [Config-Based Prediction]
 MPG : 28.02
 KMPL: 11.91

 [Brand-Based Prediction]
 Car : BMW 1 Series
 MPG : 21.35
 KMPL: 9.07
 MSRP (lookup only, not a model feature): 40650


(np.float64(21.346522429791577), np.float64(9.0744066849044))

In [ ]:

# ============================================================
# PART E: REPRODUCIBILITY INFO
# ============================================================
print("=" * 60)
print("PART E: Reproducibility Information")
print("=" * 60)

try:
    import lime as lime_pkg
    lime_version = getattr(lime_pkg, "__version__", "unknown")
except Exception:
    lime_version = "unknown"

repro_info = {
    "python_version":   sys.version.split()[0],
    "platform":         platform.platform(),
    "processor":        platform.processor() or "unknown (typical Colab runtime: Intel Xeon CPU)",
    "sklearn_version":  sklearn.__version__,
    "xgboost_version":  xgb.__version__,
    "shap_version":     shap.__version__,
    "lime_version":     lime_version,
    "numpy_version":    np.__version__,
    "pandas_version":   pd.__version__,
    "random_state":     RANDOM_STATE,
    "current_year_used_for_age_features": CURRENT_YEAR,
    "environment_note": "Google Colab (standard CPU runtime, RAM ~12-13GB unless upgraded)",
}
for k, v in repro_info.items():
    print(f"{k}: {v}")

joblib.dump(repro_info, "reproducibility_info.pkl")
with open("reproducibility_info.txt", "w") as f:
    for k, v in repro_info.items():
        f.write(f"{k}: {v}\n")

print("\nReproducibility info saved: reproducibility_info.txt / .pkl\n")


PART E: Reproducibility Information
python_version: 3.12.13
platform: Linux-6.6.122+-x86_64-with-glibc2.35
processor: x86_64
sklearn_version: 1.6.1
xgboost_version: 3.3.0
shap_version: 0.52.0
lime_version: unknown
numpy_version: 2.0.2
pandas_version: 2.2.2
random_state: 42
current_year_used_for_age_features: 2026
environment_note: Google Colab (standard CPU runtime, RAM ~12-13GB unless upgraded)

Reproducibility info saved: reproducibility_info.txt / .pkl



In [ ]:
# ============================================================
# PART F: DOWNLOAD ARTIFACTS
# ============================================================
for f in [
    "model_config.pkl", "model_config_rf.pkl", "scaler_config.pkl", "features_config.pkl",
    "model_xgb_brand.pkl", "model_rf_brand.pkl", "scaler_brand.pkl", "features_brand.pkl",
    "brand_metrics.pkl",
    "actual_vs_predicted_config.png", "residual_scatter_config.png", "residual_distribution_config.png",
    "actual_vs_predicted_brand.png", "residual_scatter_brand.png", "residual_distribution_brand.png",
    "tree_importance_comparison_config.png", "tree_importance_comparison_brand.png",
    "shap_summary_config.png", "shap_bar_config.png", "shap_local_config_instance0.png",
    "shap_summary_brand.png", "shap_bar_brand.png", "shap_local_brand_instance0.png",
    "lime_local_config_instance0.png", "lime_local_config_instance0.html",
    "lime_local_brand_instance0.png", "lime_local_brand_instance0.html",
    "shap_lime_comparison_config.png", "shap_lime_comparison_brand.png",
    "xai_comparison_results.pkl",
    "reproducibility_info.txt", "reproducibility_info.pkl",
]:
    files.download(f)

print("All artifacts downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

All artifacts downloaded!


In [ ]:
for f in [
 "shap_bar_brand.png",
]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
"""
FEATURE ENGINEERING ABLATION STUDY
=====================================
Run this AFTER your existing Part A (Config) and Part B (Brand) sections
in the same Colab session — it reuses df, df2, scaler objects, etc.
already in memory. Produces the exact numbers for your ablation table.
"""

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import xgboost as xgb
import numpy as np
import pandas as pd

# ============================================================
# ABLATION 1: CONFIGURATION MODEL (Auto MPG)
# ============================================================
print("=" * 60)
print("ABLATION — Configuration Model")
print("=" * 60)

# Baseline: raw Auto MPG columns only (no engineered features)
raw_features_config = ["cylinders", "displacement", "horsepower", "weight"]

# Full: raw + engineered (power_to_weight, car_age, hp_per_cylinder)
full_features_config = [
    "cylinders", "displacement", "horsepower", "weight",
    "power_to_weight", "car_age", "hp_per_cylinder"
]

def run_ablation(df, feature_list, target_col, label, random_state=42):
    X = df[feature_list]
    y = df[target_col]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state
    )
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    model = xgb.XGBRegressor(
        n_estimators=200, learning_rate=0.05, max_depth=6,
        subsample=0.8, random_state=random_state, verbosity=0
    )
    model.fit(X_train_sc, y_train)
    preds = model.predict(X_test_sc)

    r2   = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae  = mean_absolute_error(y_test, preds)
    print(f"{label:35s} | R2: {r2:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f}")
    return r2, rmse, mae

r2_base_c, rmse_base_c, mae_base_c = run_ablation(
    df, raw_features_config, "mpg", "Config - Baseline (raw only)"
)
r2_full_c, rmse_full_c, mae_full_c = run_ablation(
    df, full_features_config, "mpg", "Config - Full (raw + engineered)"
)

print(f"\nDifference (Full - Baseline):")
print(f"R2:   {r2_full_c - r2_base_c:+.4f}")
print(f"RMSE: {rmse_full_c - rmse_base_c:+.4f}")
print(f"MAE:  {mae_full_c - mae_base_c:+.4f}")


# ============================================================
# ABLATION 2: BRAND MODEL (Car Specifications)
# ============================================================
print("\n" + "=" * 60)
print("ABLATION — Brand Model")
print("=" * 60)

# Baseline: raw Car Specs columns only
raw_features_brand = ["Engine HP", "Engine Cylinders"]

# Full: raw + engineered (weight_est, vehicle_age, power_to_weight, hp_per_cylinder)
full_features_brand = [
    "Engine HP", "Engine Cylinders", "weight_est",
    "vehicle_age", "power_to_weight", "hp_per_cylinder"
]

r2_base_b, rmse_base_b, mae_base_b = run_ablation(
    df2, raw_features_brand, "avg_mpg", "Brand - Baseline (raw only)"
)
r2_full_b, rmse_full_b, mae_full_b = run_ablation(
    df2, full_features_brand, "avg_mpg", "Brand - Full (raw + engineered)"
)

print(f"\nDifference (Full - Baseline):")
print(f"R2:   {r2_full_b - r2_base_b:+.4f}")
print(f"RMSE: {rmse_full_b - rmse_base_b:+.4f}")
print(f"MAE:  {mae_full_b - mae_base_b:+.4f}")

print("\n" + "=" * 60)
print("COPY THESE 4 NUMBERS (R2/RMSE/MAE for each row) BACK TO CLAUDE")
print("=" * 60)

ABLATION — Configuration Model
Config - Baseline (raw only)        | R2: 0.7051 | RMSE: 3.9113 | MAE: 2.7451
Config - Full (raw + engineered)    | R2: 0.8400 | RMSE: 2.8809 | MAE: 1.9802

Difference (Full - Baseline):
R2:   +0.1349
RMSE: -1.0304
MAE:  -0.7649

ABLATION — Brand Model
Brand - Baseline (raw only)         | R2: 0.7639 | RMSE: 2.8936 | MAE: 2.0580
Brand - Full (raw + engineered)     | R2: 0.8673 | RMSE: 2.1693 | MAE: 1.4937

Difference (Full - Baseline):
R2:   +0.1034
RMSE: -0.7243
MAE:  -0.5643

COPY THESE 4 NUMBERS (R2/RMSE/MAE for each row) BACK TO CLAUDE


In [ ]:
import os
files_needed = [
    "model_xgb_final.pkl", "model_rf_final.pkl", "scaler_final.pkl", "model_metrics.pkl",
    "model_xgb_brand.pkl", "model_rf_brand.pkl", "scaler_brand_final.pkl", "brand_metrics.pkl",
    "X_test_data.csv", "X_train_data.csv", "X_brand_test.csv", "X_brand_train.csv",
    "auto-mpg.csv"
]
for f in files_needed:
    print(f"{'✅' if os.path.exists(f) else '❌'} {f}")

✅ model_xgb_final.pkl
✅ model_rf_final.pkl
✅ scaler_final.pkl
✅ model_metrics.pkl
✅ model_xgb_brand.pkl
✅ model_rf_brand.pkl
✅ scaler_brand_final.pkl
✅ brand_metrics.pkl
✅ X_test_data.csv
✅ X_train_data.csv
✅ X_brand_test.csv
✅ X_brand_train.csv
✅ auto-mpg.csv


In [ ]:
!pip install lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=8dcfe11a9b2f74a57c2fd748cbcee969020459c8a767a78748c60f9eada9cdc1
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


In [2]:
import lime
import lime.lime_tabular

print("LIME installed successfully!")

LIME installed successfully!


In [3]:
# ============================================================
# JOURNAL PUBLICATION FIGURES — FIGURES 1–12
# ============================================================
# FINAL FORMAT:
#   ONE FILE PER FIGURE
#   TIFF
#   COLOUR
#   600 DPI
#   GRAYSCALE-SAFE
#   NO UNNECESSARY BOUNDARY/SPINE LINES
#
# Journal requirement:
# PNG/EPS/TIFF/JPEG/BMP
# Grayscale >= 600 DPI
# Colour >= 300 DPI
#
# We use ONE colour TIFF at 600 DPI for every figure.
# ============================================================


# ============================================================
# 0. INSTALL REQUIRED PACKAGE
# ============================================================

!pip -q install lime


# ============================================================
# 1. IMPORTS
# ============================================================

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import joblib
import os
import shutil
import zipfile

from sklearn.model_selection import train_test_split


# ============================================================
# 2. GLOBAL PUBLICATION STYLE
# ============================================================

plt.rcParams.update({

    # Publication font
    "font.family": "serif",

    # General text
    "font.size": 11,

    # Titles
    "axes.titlesize": 12,
    "axes.titleweight": "bold",

    # Axis labels
    "axes.labelsize": 11,

    # Tick labels
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,

    # Figure
    "figure.dpi": 150,

    # Saving
    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.08,

    # White background
    "figure.facecolor": "white",
    "axes.facecolor": "white"
})


# ============================================================
# 3. JOURNAL SETTINGS
# ============================================================

FINAL_DPI = 600
OUTPUT_DIR = "Journal_Figures"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# 4. COLOUR PALETTE
#    Also use hatch/marker differences so figures remain
#    understandable in grayscale.
# ============================================================

MODEL_COLORS = [
    "#2196F3",   # XGBoost
    "#F44336",   # Random Forest
    "#4CAF50"    # Ensemble
]

MODEL_HATCHES = [
    "///",
    "\\\\\\",
    "xxx"
]

MODEL_MARKERS = [
    "o",
    "s",
    "^"
]


# ============================================================
# 5. HELPER FUNCTIONS
# ============================================================

def remove_all_spines(ax):
    """
    Remove all rectangular boundary lines.
    """
    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.tick_params(
        axis="both",
        which="both",
        length=3,
        width=0.7
    )


def clean_legend(legend):
    """
    Remove unnecessary legend frame/border.
    """
    if legend is not None:
        legend.get_frame().set_linewidth(0)
        legend.get_frame().set_edgecolor("none")
        legend.get_frame().set_facecolor("white")


def save_journal_figure(fig, filename):
    """
    Save ONE final colour TIFF at 600 DPI.

    This is:
        Colour >= 300 DPI  -> YES
        600 DPI             -> YES
        Print quality       -> YES
        TIFF format         -> YES
    """

    filepath = os.path.join(
        OUTPUT_DIR,
        filename
    )

    fig.savefig(
        filepath,
        dpi=FINAL_DPI,
        format="tiff",
        bbox_inches="tight",
        pad_inches=0.08,
        facecolor="white",
        edgecolor="none",
        pil_kwargs={
            "compression": "tiff_lzw"
        }
    )

    plt.close(fig)

    print(f"✅ Saved: {filepath}")

    return filepath


# ============================================================
# 6. LOAD MODELS
# ============================================================

model_xgb_c = joblib.load("model_xgb_final.pkl")
model_rf_c  = joblib.load("model_rf_final.pkl")
scaler_c    = joblib.load("scaler_final.pkl")
metrics_c   = joblib.load("model_metrics.pkl")

model_xgb_b = joblib.load("model_xgb_brand.pkl")
model_rf_b  = joblib.load("model_rf_brand.pkl")
scaler_b    = joblib.load("scaler_brand_final.pkl")
metrics_b   = joblib.load("brand_metrics.pkl")


# ============================================================
# 7. LOAD TEST / TRAIN DATA
# ============================================================

X_test_c  = pd.read_csv("X_test_data.csv")
X_train_c = pd.read_csv("X_train_data.csv")

X_test_b  = pd.read_csv("X_brand_test.csv")
X_train_b = pd.read_csv("X_brand_train.csv")


# ============================================================
# 8. FEATURE NAMES
# ============================================================

config_features = [
    "cylinders",
    "displacement",
    "horsepower",
    "weight",
    "model-year",
    "power_to_weight",
    "car_age",
    "hp_per_cylinder"
]

brand_features = [
    "Engine HP",
    "Engine Cylinders",
    "weight_est",
    "displacement_est",
    "vehicle_age",
    "power_to_weight",
    "hp_per_cylinder"
]


# ============================================================
# 9. ENSEMBLE WEIGHTS
# ============================================================

XGB_W = 0.60
RF_W  = 0.40


# ============================================================
# 10. PREDICTIONS
# ============================================================

y_pred_xgb_c = model_xgb_c.predict(X_test_c.values)
y_pred_rf_c  = model_rf_c.predict(X_test_c.values)

y_pred_ens_c = (
    XGB_W * y_pred_xgb_c
    +
    RF_W * y_pred_rf_c
)


y_pred_xgb_b = model_xgb_b.predict(X_test_b.values)
y_pred_rf_b  = model_rf_b.predict(X_test_b.values)

y_pred_ens_b = (
    XGB_W * y_pred_xgb_b
    +
    RF_W * y_pred_rf_b
)


# ============================================================
# 11. LOAD ORIGINAL TARGET
# ============================================================

df_orig = pd.read_csv("auto-mpg.csv")

df_orig["horsepower"] = (
    df_orig["horsepower"]
    .replace("?", None)
    .astype(float)
)

df_orig = df_orig.dropna()

df_orig["power_to_weight"] = (
    df_orig["horsepower"] /
    df_orig["weight"]
)

df_orig["car_age"] = (
    2026 -
    df_orig["model-year"]
)

df_orig["hp_per_cylinder"] = (
    df_orig["horsepower"] /
    df_orig["cylinders"]
)

X_c = df_orig[config_features]
y_c = df_orig["mpg"]

_, X_c_test, _, y_c_test = train_test_split(
    X_c,
    y_c,
    test_size=0.2,
    random_state=42
)


# ============================================================
# FIGURE 1
# MODEL PERFORMANCE — CONFIGURATION DATASET
# ============================================================

models_list = [
    "XGBoost",
    "Random Forest",
    "Ensemble"
]

r2_c = [
    metrics_c["xgb_r2"],
    metrics_c["rf_r2"],
    metrics_c["ens_r2"]
]

rmse_c = [
    metrics_c["xgb_rmse"],
    metrics_c["rf_rmse"],
    metrics_c["ens_rmse"]
]


fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 4.2)
)


# -------------------------
# Figure 1(a)
# -------------------------

bars = axes[0].bar(
    models_list,
    r2_c,
    color=MODEL_COLORS,
    edgecolor="black",
    linewidth=0.6
)

for bar, hatch in zip(bars, MODEL_HATCHES):
    bar.set_hatch(hatch)

axes[0].set_title("R² score")
axes[0].set_ylabel("R² score")
axes[0].set_ylim(0.80, 0.95)

remove_all_spines(axes[0])


# -------------------------
# Figure 1(b)
# -------------------------

bars = axes[1].bar(
    models_list,
    rmse_c,
    color=MODEL_COLORS,
    edgecolor="black",
    linewidth=0.6
)

for bar, hatch in zip(bars, MODEL_HATCHES):
    bar.set_hatch(hatch)

axes[1].set_title("RMSE")
axes[1].set_ylabel("RMSE")

remove_all_spines(axes[1])


fig.suptitle(
    "Model performance comparison for the configuration dataset",
    fontsize=13,
    fontweight="bold"
)

fig.text(
    0.25,
    -0.01,
    "(a)",
    ha="center",
    fontsize=10
)

fig.text(
    0.75,
    -0.01,
    "(b)",
    ha="center",
    fontsize=10
)

fig.tight_layout()

save_journal_figure(
    fig,
    "Figure_1_Model_Performance_Config.tiff"
)


# ============================================================
# FIGURE 2
# MODEL PERFORMANCE — BRAND DATASET
# ============================================================

r2_b = [
    metrics_b["xgb_r2"],
    metrics_b["rf_r2"],
    metrics_b["ens_r2"]
]

rmse_b = [
    metrics_b["xgb_rmse"],
    metrics_b["rf_rmse"],
    metrics_b["ens_rmse"]
]


fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 4.2)
)


# Figure 2(a)

bars = axes[0].bar(
    models_list,
    r2_b,
    color=MODEL_COLORS,
    edgecolor="black",
    linewidth=0.6
)

for bar, hatch in zip(bars, MODEL_HATCHES):
    bar.set_hatch(hatch)

axes[0].set_title("R² score")
axes[0].set_ylabel("R² score")
axes[0].set_ylim(0.80, 0.95)

remove_all_spines(axes[0])


# Figure 2(b)

bars = axes[1].bar(
    models_list,
    rmse_b,
    color=MODEL_COLORS,
    edgecolor="black",
    linewidth=0.6
)

for bar, hatch in zip(bars, MODEL_HATCHES):
    bar.set_hatch(hatch)

axes[1].set_title("RMSE")
axes[1].set_ylabel("RMSE")

remove_all_spines(axes[1])


fig.suptitle(
    "Model performance comparison for the brand dataset",
    fontsize=13,
    fontweight="bold"
)

fig.text(
    0.25,
    -0.01,
    "(a)",
    ha="center",
    fontsize=10
)

fig.text(
    0.75,
    -0.01,
    "(b)",
    ha="center",
    fontsize=10
)

fig.tight_layout()

save_journal_figure(
    fig,
    "Figure_2_Model_Performance_Brand.tiff"
)


# ============================================================
# FIGURE 3
# ACTUAL VS PREDICTED
# ============================================================

titles = [
    "XGBoost",
    "Random Forest",
    "Ensemble"
]

preds = [
    y_pred_xgb_c,
    y_pred_rf_c,
    y_pred_ens_c
]


fig, axes = plt.subplots(
    1,
    3,
    figsize=(13.5, 4.2)
)


for i, (ax, pred, title, col, marker) in enumerate(
    zip(
        axes,
        preds,
        titles,
        MODEL_COLORS,
        MODEL_MARKERS
    )
):

    ax.scatter(
        y_c_test,
        pred,
        alpha=0.55,
        color=col,
        marker=marker,
        s=24,
        edgecolors="black",
        linewidths=0.25,
        label=title
    )

    mn = min(
        y_c_test.min(),
        pred.min()
    )

    mx = max(
        y_c_test.max(),
        pred.max()
    )

    ax.plot(
        [mn, mx],
        [mn, mx],
        "k--",
        linewidth=1.1,
        label="Perfect fit"
    )

    ax.set_xlabel("Actual MPG")
    ax.set_ylabel("Predicted MPG")
    ax.set_title(title)

    legend = ax.legend(
        fontsize=8,
        frameon=False
    )

    clean_legend(legend)

    remove_all_spines(ax)


fig.suptitle(
    "Actual versus predicted MPG for the configuration dataset",
    fontsize=13,
    fontweight="bold"
)

fig.text(
    0.18,
    -0.01,
    "(a)",
    ha="center"
)

fig.text(
    0.50,
    -0.01,
    "(b)",
    ha="center"
)

fig.text(
    0.82,
    -0.01,
    "(c)",
    ha="center"
)

fig.tight_layout()

save_journal_figure(
    fig,
    "Figure_3_Actual_vs_Predicted.tiff"
)


# ============================================================
# FIGURE 4
# RESIDUAL ANALYSIS
# ============================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(13.5, 4.2)
)


for ax, pred, title, col, marker in zip(
    axes,
    preds,
    titles,
    MODEL_COLORS,
    MODEL_MARKERS
):

    residuals = (
        y_c_test.values -
        pred
    )

    ax.scatter(
        pred,
        residuals,
        alpha=0.55,
        color=col,
        marker=marker,
        s=24,
        edgecolors="black",
        linewidths=0.25
    )

    ax.axhline(
        y=0,
        color="black",
        linestyle="--",
        linewidth=1
    )

    ax.set_xlabel("Predicted MPG")
    ax.set_ylabel("Residuals")
    ax.set_title(title)

    remove_all_spines(ax)


fig.suptitle(
    "Residual analysis for the configuration dataset",
    fontsize=13,
    fontweight="bold"
)

fig.text(0.18, -0.01, "(a)", ha="center")
fig.text(0.50, -0.01, "(b)", ha="center")
fig.text(0.82, -0.01, "(c)", ha="center")

fig.tight_layout()

save_journal_figure(
    fig,
    "Figure_4_Residual_Analysis.tiff"
)


# ============================================================
# FIGURE 5
# FEATURE IMPORTANCE
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 4.8)
)


# -------------------------
# Config dataset
# -------------------------

x = np.arange(
    len(config_features)
)

width = 0.35

bars1 = axes[0].bar(
    x - width / 2,
    model_xgb_c.feature_importances_,
    width,
    label="XGBoost",
    color=MODEL_COLORS[0],
    hatch=MODEL_HATCHES[0],
    edgecolor="black",
    linewidth=0.5
)

bars2 = axes[0].bar(
    x + width / 2,
    model_rf_c.feature_importances_,
    width,
    label="Random Forest",
    color=MODEL_COLORS[1],
    hatch=MODEL_HATCHES[1],
    edgecolor="black",
    linewidth=0.5
)

axes[0].set_xticks(x)
axes[0].set_xticklabels(
    config_features,
    rotation=45,
    ha="right",
    fontsize=8
)

axes[0].set_ylabel(
    "Feature importance"
)

axes[0].set_title(
    "Configuration dataset"
)

legend = axes[0].legend(
    frameon=False
)

clean_legend(legend)

remove_all_spines(axes[0])


# -------------------------
# Brand dataset
# -------------------------

x2 = np.arange(
    len(brand_features)
)

bars3 = axes[1].bar(
    x2 - width / 2,
    model_xgb_b.feature_importances_,
    width,
    label="XGBoost",
    color=MODEL_COLORS[0],
    hatch=MODEL_HATCHES[0],
    edgecolor="black",
    linewidth=0.5
)

bars4 = axes[1].bar(
    x2 + width / 2,
    model_rf_b.feature_importances_,
    width,
    label="Random Forest",
    color=MODEL_COLORS[1],
    hatch=MODEL_HATCHES[1],
    edgecolor="black",
    linewidth=0.5
)

axes[1].set_xticks(x2)
axes[1].set_xticklabels(
    brand_features,
    rotation=45,
    ha="right",
    fontsize=8
)

axes[1].set_ylabel(
    "Feature importance"
)

axes[1].set_title(
    "Brand dataset"
)

legend = axes[1].legend(
    frameon=False
)

clean_legend(legend)

remove_all_spines(axes[1])


fig.suptitle(
    "Feature importance of XGBoost and Random Forest models",
    fontsize=13,
    fontweight="bold"
)

fig.text(0.25, -0.01, "(a)", ha="center")
fig.text(0.75, -0.01, "(b)", ha="center")

fig.tight_layout()

save_journal_figure(
    fig,
    "Figure_5_Feature_Importance.tiff"
)


# ============================================================
# FIGURE 6
# SHAP SUMMARY — CONFIGURATION DATASET
# ============================================================

explainer_c = shap.TreeExplainer(
    model_xgb_c
)

shap_vals_c = explainer_c.shap_values(
    X_test_c.values
)


fig, axes = plt.subplots(
    1,
    2,
    figsize=(13.5, 5)
)


# -------------------------
# SHAP bar
# -------------------------

plt.sca(axes[0])

shap.summary_plot(
    shap_vals_c,
    X_test_c,
    feature_names=config_features,
    plot_type="bar",
    show=False
)

axes[0].set_title(
    "SHAP feature importance"
)

remove_all_spines(axes[0])


# -------------------------
# SHAP beeswarm
# -------------------------

plt.sca(axes[1])

shap.summary_plot(
    shap_vals_c,
    X_test_c,
    feature_names=config_features,
    show=False
)

axes[1].set_title(
    "SHAP feature impact"
)

remove_all_spines(axes[1])


fig.suptitle(
    "SHAP analysis for the configuration dataset",
    fontsize=13,
    fontweight="bold"
)

fig.text(0.25, -0.01, "(a)", ha="center")
fig.text(0.75, -0.01, "(b)", ha="center")

fig.tight_layout()

save_journal_figure(
    fig,
    "Figure_6_SHAP_Summary_Config.tiff"
)


# ============================================================
# FIGURE 7
# SHAP SUMMARY — BRAND DATASET
# ============================================================

explainer_b = shap.TreeExplainer(
    model_xgb_b
)

shap_vals_b = explainer_b.shap_values(
    X_test_b.values
)


fig, axes = plt.subplots(
    1,
    2,
    figsize=(13.5, 5)
)


# SHAP bar

plt.sca(axes[0])

shap.summary_plot(
    shap_vals_b,
    X_test_b,
    feature_names=brand_features,
    plot_type="bar",
    show=False
)

axes[0].set_title(
    "SHAP feature importance"
)

remove_all_spines(axes[0])


# SHAP beeswarm

plt.sca(axes[1])

shap.summary_plot(
    shap_vals_b,
    X_test_b,
    feature_names=brand_features,
    show=False
)

axes[1].set_title(
    "SHAP feature impact"
)

remove_all_spines(axes[1])


fig.suptitle(
    "SHAP analysis for the brand dataset",
    fontsize=13,
    fontweight="bold"
)

fig.text(0.25, -0.01, "(a)", ha="center")
fig.text(0.75, -0.01, "(b)", ha="center")

fig.tight_layout()

save_journal_figure(
    fig,
    "Figure_7_SHAP_Summary_Brand.tiff"
)


# ============================================================
# FIGURE 8
# SHAP WATERFALL — CONFIGURATION DATASET
# ============================================================

instance_c = X_test_c.values[0:1]

sv_c = explainer_c.shap_values(
    instance_c
)

shap_exp_c = shap.Explanation(
    values=sv_c[0],
    base_values=explainer_c.expected_value,
    data=instance_c[0],
    feature_names=config_features
)


fig = plt.figure(
    figsize=(8, 5)
)

shap.plots.waterfall(
    shap_exp_c,
    show=False
)

plt.title(
    "SHAP local explanation for the configuration dataset",
    fontsize=12,
    fontweight="bold"
)

# Remove surrounding boundary lines
ax = plt.gca()
remove_all_spines(ax)

plt.tight_layout()

save_journal_figure(
    fig,
    "Figure_8_SHAP_Waterfall_Config.tiff"
)


# ============================================================
# FIGURE 9
# SHAP WATERFALL — BRAND DATASET
# ============================================================

instance_b = X_test_b.values[0:1]

sv_b = explainer_b.shap_values(
    instance_b
)

shap_exp_b = shap.Explanation(
    values=sv_b[0],
    base_values=explainer_b.expected_value,
    data=instance_b[0],
    feature_names=brand_features
)


fig = plt.figure(
    figsize=(8, 5)
)

shap.plots.waterfall(
    shap_exp_b,
    show=False
)

plt.title(
    "SHAP local explanation for the brand dataset",
    fontsize=12,
    fontweight="bold"
)

ax = plt.gca()
remove_all_spines(ax)

plt.tight_layout()

save_journal_figure(
    fig,
    "Figure_9_SHAP_Waterfall_Brand.tiff"
)


# ============================================================
# FIGURE 10
# LIME — CONFIGURATION DATASET
# ============================================================

import lime
import lime.lime_tabular


lime_exp_c = lime.lime_tabular.LimeTabularExplainer(
    X_train_c.values,
    feature_names=config_features,
    mode="regression"
)


def ensemble_pred_c(x):

    return (
        XGB_W *
        model_xgb_c.predict(x)
        +
        RF_W *
        model_rf_c.predict(x)
    )


exp_c = lime_exp_c.explain_instance(
    instance_c[0],
    ensemble_pred_c,
    num_features=8
)

lime_c = exp_c.as_list()

lime_f_c = [
    item[0]
    for item in lime_c
]

lime_w_c = [
    item[1]
    for item in lime_c
]


fig, ax = plt.subplots(
    figsize=(8, 5)
)


lime_colors_c = [
    MODEL_COLORS[2] if w > 0
    else MODEL_COLORS[1]
    for w in lime_w_c
]

lime_hatches_c = [
    "///" if w > 0
    else "\\\\\\"
    for w in lime_w_c
]


bars = ax.barh(
    lime_f_c,
    lime_w_c,
    color=lime_colors_c,
    edgecolor="black",
    linewidth=0.5
)

for bar, hatch in zip(
    bars,
    lime_hatches_c
):
    bar.set_hatch(hatch)


ax.axvline(
    x=0,
    color="black",
    linewidth=1
)

ax.set_xlabel(
    "LIME weight"
)

ax.set_title(
    "LIME explanation for the configuration dataset",
    fontsize=12,
    fontweight="bold"
)

remove_all_spines(ax)

plt.tight_layout()

save_journal_figure(
    fig,
    "Figure_10_LIME_Config.tiff"
)


# ============================================================
# FIGURE 11
# LIME — BRAND DATASET
# ============================================================

lime_exp_b = lime.lime_tabular.LimeTabularExplainer(
    X_train_b.values,
    feature_names=brand_features,
    mode="regression"
)


def ensemble_pred_b(x):

    return (
        XGB_W *
        model_xgb_b.predict(x)
        +
        RF_W *
        model_rf_b.predict(x)
    )


exp_b = lime_exp_b.explain_instance(
    instance_b[0],
    ensemble_pred_b,
    num_features=7
)

lime_b = exp_b.as_list()

lime_f_b = [
    item[0]
    for item in lime_b
]

lime_w_b = [
    item[1]
    for item in lime_b
]


fig, ax = plt.subplots(
    figsize=(8, 5)
)


lime_colors_b = [
    MODEL_COLORS[2] if w > 0
    else MODEL_COLORS[1]
    for w in lime_w_b
]

lime_hatches_b = [
    "///" if w > 0
    else "\\\\\\"
    for w in lime_w_b
]


bars = ax.barh(
    lime_f_b,
    lime_w_b,
    color=lime_colors_b,
    edgecolor="black",
    linewidth=0.5
)

for bar, hatch in zip(
    bars,
    lime_hatches_b
):
    bar.set_hatch(hatch)


ax.axvline(
    x=0,
    color="black",
    linewidth=1
)

ax.set_xlabel(
    "LIME weight"
)

ax.set_title(
    "LIME explanation for the brand dataset",
    fontsize=12,
    fontweight="bold"
)

remove_all_spines(ax)

plt.tight_layout()

save_journal_figure(
    fig,
    "Figure_11_LIME_Brand.tiff"
)


# ============================================================
# FIGURE 12
# SHAP VS LIME — CONFIGURATION DATASET
# ============================================================

shap_vals_single_c = sv_c[0]


fig, axes = plt.subplots(
    1,
    2,
    figsize=(13.5, 5)
)


# -------------------------
# SHAP
# -------------------------

shap_colors = [
    MODEL_COLORS[2] if v > 0
    else MODEL_COLORS[1]
    for v in shap_vals_single_c
]

shap_hatches = [
    "///" if v > 0
    else "\\\\\\"
    for v in shap_vals_single_c
]


bars = axes[0].barh(
    config_features,
    shap_vals_single_c,
    color=shap_colors,
    edgecolor="black",
    linewidth=0.5
)

for bar, hatch in zip(
    bars,
    shap_hatches
):
    bar.set_hatch(hatch)


axes[0].axvline(
    x=0,
    color="black",
    linewidth=1
)

axes[0].set_title(
    "SHAP values"
)

axes[0].set_xlabel(
    "SHAP value"
)

remove_all_spines(axes[0])


# -------------------------
# LIME
# -------------------------

lime_colors = [
    MODEL_COLORS[2] if w > 0
    else MODEL_COLORS[1]
    for w in lime_w_c
]

lime_hatches = [
    "///" if w > 0
    else "\\\\\\"
    for w in lime_w_c
]


bars = axes[1].barh(
    lime_f_c,
    lime_w_c,
    color=lime_colors,
    edgecolor="black",
    linewidth=0.5
)

for bar, hatch in zip(
    bars,
    lime_hatches
):
    bar.set_hatch(hatch)


axes[1].axvline(
    x=0,
    color="black",
    linewidth=1
)

axes[1].set_title(
    "LIME weights"
)

axes[1].set_xlabel(
    "LIME weight"
)

remove_all_spines(axes[1])


fig.suptitle(
    "Comparison of SHAP and LIME local explanations",
    fontsize=13,
    fontweight="bold"
)

fig.text(
    0.25,
    -0.01,
    "(a)",
    ha="center"
)

fig.text(
    0.75,
    -0.01,
    "(b)",
    ha="center"
)

fig.tight_layout()

save_journal_figure(
    fig,
    "Figure_12_SHAP_vs_LIME.tiff"
)


# ============================================================
# 13. CREATE ZIP FILE
# ============================================================

zip_filename = "Journal_Figures_TIFF_600dpi.zip"

with zipfile.ZipFile(
    zip_filename,
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    for filename in sorted(
        os.listdir(OUTPUT_DIR)
    ):

        if filename.lower().endswith(".tiff"):

            filepath = os.path.join(
                OUTPUT_DIR,
                filename
            )

            zipf.write(
                filepath,
                arcname=filename
            )


# ============================================================
# 14. FINAL CHECK
# ============================================================

figure_files = sorted(
    [
        f
        for f in os.listdir(OUTPUT_DIR)
        if f.lower().endswith(".tiff")
    ]
)


print("\n" + "=" * 65)
print("PUBLICATION FIGURE GENERATION COMPLETE")
print("=" * 65)

print(f"\nNumber of figures generated: {len(figure_files)}")

for f in figure_files:
    print("✓", f)

print("\nFormat       : TIFF")
print("Colour       : YES")
print("Resolution   : 600 DPI")
print("Colour rule  : >= 300 DPI ✓")
print("Print quality : HIGH")
print("Spines       : REMOVED")
print("Grayscale-safe design: YES")
print("\nZIP created:")
print(zip_filename)

print("\n" + "=" * 65)


# ============================================================
# 15. DOWNLOAD ONE ZIP
# ============================================================

from google.colab import files

files.download(zip_filename)

✅ Saved: Journal_Figures/Figure_1_Model_Performance_Config.tiff
✅ Saved: Journal_Figures/Figure_2_Model_Performance_Brand.tiff
✅ Saved: Journal_Figures/Figure_3_Actual_vs_Predicted.tiff
✅ Saved: Journal_Figures/Figure_4_Residual_Analysis.tiff
✅ Saved: Journal_Figures/Figure_5_Feature_Importance.tiff
✅ Saved: Journal_Figures/Figure_6_SHAP_Summary_Config.tiff
✅ Saved: Journal_Figures/Figure_7_SHAP_Summary_Brand.tiff
✅ Saved: Journal_Figures/Figure_8_SHAP_Waterfall_Config.tiff
✅ Saved: Journal_Figures/Figure_9_SHAP_Waterfall_Brand.tiff
✅ Saved: Journal_Figures/Figure_10_LIME_Config.tiff
✅ Saved: Journal_Figures/Figure_11_LIME_Brand.tiff
✅ Saved: Journal_Figures/Figure_12_SHAP_vs_LIME.tiff

PUBLICATION FIGURE GENERATION COMPLETE

Number of figures generated: 12
✓ Figure_10_LIME_Config.tiff
✓ Figure_11_LIME_Brand.tiff
✓ Figure_12_SHAP_vs_LIME.tiff
✓ Figure_1_Model_Performance_Config.tiff
✓ Figure_2_Model_Performance_Brand.tiff
✓ Figure_3_Actual_vs_Predicted.tiff
✓ Figure_4_Residual_Analysis

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
# ============================================================
# ============================================================
# JOURNAL FINAL FIGURE GENERATION
# ============================================================
#
# ONLY PAPER FIGURES:
#
#   Figure 2  -> Actual vs Predicted MPG — Configuration
#   Figure 3  -> Actual vs Predicted avg_mpg — Brand
#   Figure 6  -> SHAP Summary — Configuration
#   Figure 7  -> SHAP Summary — Brand
#   Figure 8  -> LIME Local Explanation — Configuration
#   Figure 9  -> LIME Local Explanation — Brand
#   Figure 10 -> SHAP vs LIME — Configuration
#   Figure 11 -> SHAP vs LIME — Brand
#
# FINAL FORMAT:
#   ONE FILE PER FIGURE
#   COLOUR TIFF
#   600 DPI
#
# Journal requirement:
#   Colour >= 300 DPI
#   Grayscale >= 600 DPI
#
# A 600-DPI colour TIFF is used.
# The figures are also designed to remain interpretable
# when printed in grayscale using hatches/markers where useful.
#
# ============================================================


# ============================================================
# 0. INSTALL LIME
# ============================================================

!pip -q install lime


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import shutil
import zipfile
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import joblib
import shap
import lime
import lime.lime_tabular

from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr


# ============================================================
# 2. PUBLICATION STYLE
# ============================================================

plt.rcParams.update({

    "font.family": "serif",

    "font.size": 11,

    "axes.titlesize": 13,

    "axes.titleweight": "bold",

    "axes.labelsize": 11,

    "xtick.labelsize": 9,

    "ytick.labelsize": 9,

    "legend.fontsize": 9,

    "figure.facecolor": "white",

    "axes.facecolor": "white",

    "savefig.facecolor": "white",

    "savefig.edgecolor": "white"
})


# ============================================================
# 3. FINAL JOURNAL SETTINGS
# ============================================================

FINAL_DPI = 600

OUTPUT_DIR = "Journal_Final_Figures"

# Remove old output from previous run
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# 4. COLOUR + GRAYSCALE-SAFE PATTERNS
# ============================================================

POSITIVE_COLOR = "#4CAF50"
NEGATIVE_COLOR = "#F44336"

XGB_COLOR = "#2196F3"
RF_COLOR = "#F44336"
ENS_COLOR = "#4CAF50"

POSITIVE_HATCH = "///"
NEGATIVE_HATCH = "\\\\\\"


# ============================================================
# 5. HELPER FUNCTIONS
# ============================================================

def remove_spines(ax):
    """
    Remove all rectangular boundary lines.
    """

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.tick_params(
        axis="both",
        which="both",
        length=3,
        width=0.7
    )


def clean_legend(legend):
    """
    Remove legend border.
    """

    if legend is not None:
        legend.get_frame().set_linewidth(0)
        legend.get_frame().set_edgecolor("none")
        legend.get_frame().set_facecolor("white")


def save_tiff(fig, filename):
    """
    Save ONE final colour TIFF at 600 DPI.
    """

    path = os.path.join(
        OUTPUT_DIR,
        filename
    )

    fig.savefig(
        path,
        dpi=FINAL_DPI,
        format="tiff",
        bbox_inches="tight",
        pad_inches=0.08,
        facecolor="white",
        edgecolor="none",
        pil_kwargs={
            "compression": "tiff_lzw"
        }
    )

    plt.close(fig)

    print(f"✅ Saved: {filename}")

    return path


# ============================================================
# 6. LOAD YOUR FINAL MODELS
# ============================================================

model_xgb_c = joblib.load(
    "model_xgb_final.pkl"
)

model_rf_c = joblib.load(
    "model_rf_final.pkl"
)

scaler_c = joblib.load(
    "scaler_final.pkl"
)

metrics_c = joblib.load(
    "model_metrics.pkl"
)

model_xgb_b = joblib.load(
    "model_xgb_brand.pkl"
)

model_rf_b = joblib.load(
    "model_rf_brand.pkl"
)

scaler_b = joblib.load(
    "scaler_brand_final.pkl"
)

metrics_b = joblib.load(
    "brand_metrics.pkl"
)


# ============================================================
# 7. LOAD TRAIN / TEST FEATURES
# ============================================================

X_test_c = pd.read_csv(
    "X_test_data.csv"
)

X_train_c = pd.read_csv(
    "X_train_data.csv"
)

X_test_b = pd.read_csv(
    "X_brand_test.csv"
)

X_train_b = pd.read_csv(
    "X_brand_train.csv"
)


# ============================================================
# 8. FEATURE NAMES
# ============================================================

config_features = [
    "cylinders",
    "displacement",
    "horsepower",
    "weight",
    "model-year",
    "power_to_weight",
    "car_age",
    "hp_per_cylinder"
]


brand_features = [
    "Engine HP",
    "Engine Cylinders",
    "weight_est",
    "displacement_est",
    "vehicle_age",
    "power_to_weight",
    "hp_per_cylinder"
]


# ============================================================
# 9. ENSEMBLE WEIGHTS
# ============================================================

XGB_W = 0.60

RF_W = 0.40


# ============================================================
# 10. PREDICTIONS
# ============================================================

# Configuration

y_pred_xgb_c = model_xgb_c.predict(
    X_test_c.values
)

y_pred_rf_c = model_rf_c.predict(
    X_test_c.values
)

y_pred_ens_c = (
    XGB_W * y_pred_xgb_c
    +
    RF_W * y_pred_rf_c
)


# Brand

y_pred_xgb_b = model_xgb_b.predict(
    X_test_b.values
)

y_pred_rf_b = model_rf_b.predict(
    X_test_b.values
)

y_pred_ens_b = (
    XGB_W * y_pred_xgb_b
    +
    RF_W * y_pred_rf_b
)


# ============================================================
# 11. CONFIGURATION ACTUAL TARGET
# ============================================================

df_orig = pd.read_csv(
    "auto-mpg.csv"
)

df_orig["horsepower"] = (
    df_orig["horsepower"]
    .replace("?", np.nan)
    .astype(float)
)

df_orig = df_orig.dropna()

df_orig["power_to_weight"] = (
    df_orig["horsepower"]
    /
    df_orig["weight"]
)

df_orig["car_age"] = (
    2026
    -
    df_orig["model-year"]
)

df_orig["hp_per_cylinder"] = (
    df_orig["horsepower"]
    /
    df_orig["cylinders"]
)


X_c = df_orig[
    config_features
]

y_c = df_orig[
    "mpg"
]


_, X_c_test, _, y_c_test = train_test_split(
    X_c,
    y_c,
    test_size=0.20,
    random_state=42
)

y_c_test = np.asarray(
    y_c_test
)


# ============================================================
# 12. BRAND ACTUAL TARGET
# ============================================================
#
# First try to find an already-saved y-test file.
# If it exists, use it.
#
# Otherwise reconstruct avg_mpg from the original
# Car Specifications dataset.
#
# avg_mpg:
#   0.55 * Highway MPG
# + 0.45 * City MPG
#
# This follows the target construction used for
# the Brand pathway.
# ============================================================


def load_brand_actual_target():

    # --------------------------------------------------------
    # OPTION 1 — Search for saved target file
    # --------------------------------------------------------

    possible_target_files = [

        "y_brand_test.csv",
        "y_test_brand.csv",
        "y_test_b.csv",
        "brand_y_test.csv",
        "y_brand.csv",
        "brand_target_test.csv",
        "y_test_brand_data.csv"
    ]


    for filename in possible_target_files:

        if os.path.exists(filename):

            temp = pd.read_csv(
                filename
            )

            # Find numeric column
            numeric_cols = (
                temp.select_dtypes(
                    include=[np.number]
                )
                .columns
            )

            if len(numeric_cols) > 0:

                y_values = (
                    temp[numeric_cols[0]]
                    .values
                )

                if len(y_values) == len(
                    X_test_b
                ):

                    print(
                        f"✅ Brand target loaded from: "
                        f"{filename}"
                    )

                    return y_values


    # --------------------------------------------------------
    # OPTION 2 — Search common Car Specifications files
    # --------------------------------------------------------

    possible_source_files = [

        "Car Features and MSRP.csv",

        "car_features_and_msrp.csv",

        "cardataset.csv",

        "car_data.csv",

        "Car_Data.csv",

        "data.csv",

        "cars.csv",

        "vehicles.csv"
    ]


    source_file = None

    for filename in possible_source_files:

        if os.path.exists(filename):

            try:

                temp = pd.read_csv(
                    filename,
                    nrows=5
                )

                cols_lower = [
                    str(c).lower()
                    for c in temp.columns
                ]

                if (
                    "highway mpg" in cols_lower
                    and
                    "city mpg" in cols_lower
                ):

                    source_file = filename

                    break

            except:

                pass


    # --------------------------------------------------------
    # OPTION 3 — Search every CSV in current directory
    # --------------------------------------------------------

    if source_file is None:

        for filename in os.listdir("."):

            if (
                filename.lower().endswith(".csv")
                and
                filename not in [
                    "X_test_data.csv",
                    "X_train_data.csv",
                    "X_brand_test.csv",
                    "X_brand_train.csv",
                    "auto-mpg.csv"
                ]
            ):

                try:

                    temp = pd.read_csv(
                        filename,
                        nrows=5
                    )

                    cols_lower = [
                        str(c).lower()
                        for c in temp.columns
                    ]

                    if (
                        "highway mpg" in cols_lower
                        and
                        "city mpg" in cols_lower
                    ):

                        source_file = filename

                        break

                except:

                    pass


    # --------------------------------------------------------
    # Stop if source cannot be found
    # --------------------------------------------------------

    if source_file is None:

        raise FileNotFoundError(
            "\n\n❌ Could not find the Brand "
            "target data.\n\n"
            "Your X_brand_test.csv contains only "
            "the predictor features.\n\n"
            "Please place the original Car Specifications "
            "CSV OR y_brand_test.csv in the Colab directory."
        )


    print(
        f"✅ Brand source dataset found: "
        f"{source_file}"
    )


    df_b = pd.read_csv(
        source_file
    )


    # --------------------------------------------------------
    # Standardize column lookup
    # --------------------------------------------------------

    col_map = {
        str(c).strip().lower(): c
        for c in df_b.columns
    }


    highway_col = col_map.get(
        "highway mpg"
    )

    city_col = col_map.get(
        "city mpg"
    )

    engine_hp_col = col_map.get(
        "engine hp"
    )

    engine_cyl_col = col_map.get(
        "engine cylinders"
    )

    year_col = col_map.get(
        "year"
    )


    if (
        highway_col is None
        or
        city_col is None
        or
        engine_hp_col is None
        or
        engine_cyl_col is None
        or
        year_col is None
    ):

        raise ValueError(
            "❌ Required Brand dataset columns "
            "were not found."
        )


    # --------------------------------------------------------
    # Clean exactly the required fields
    # --------------------------------------------------------

    required = [
        highway_col,
        city_col,
        engine_hp_col,
        engine_cyl_col,
        year_col
    ]

    for col in required:

        df_b[col] = pd.to_numeric(
            df_b[col],
            errors="coerce"
        )


    df_b = df_b.dropna(
        subset=required
    )


    # Remove invalid / zero values
    for col in required:

        df_b = df_b[
            df_b[col] > 0
        ]


    # --------------------------------------------------------
    # Create avg_mpg
    # --------------------------------------------------------

    df_b["avg_mpg"] = (
        0.55 * df_b[highway_col]
        +
        0.45 * df_b[city_col]
    )


    y_brand = df_b[
        "avg_mpg"
    ]


    # --------------------------------------------------------
    # Reproduce 80/20 split
    # --------------------------------------------------------

    _, _, _, y_b_test = train_test_split(
        df_b,
        y_brand,
        test_size=0.20,
        random_state=42
    )


    y_b_test = np.asarray(
        y_b_test
    )


    if len(y_b_test) != len(
        X_test_b
    ):

        raise ValueError(
            "\n❌ Brand target length mismatch.\n\n"
            f"X_brand_test rows = {len(X_test_b)}\n"
            f"Recovered target rows = {len(y_b_test)}\n\n"
            "This means the original Brand preprocessing/split "
            "is different from the reconstruction."
        )


    return y_b_test


y_b_test = load_brand_actual_target()


print(
    f"✅ Brand actual target loaded: "
    f"{len(y_b_test)} test samples"
)


# ============================================================
# ============================================================
# FIGURE 2
# ACTUAL VS PREDICTED — CONFIGURATION MODEL
# ============================================================
#
# Paper:
# "Actual vs. Predicted mpg for the Configuration Model
#  (XGBoost, held-out test set)."
# ============================================================


fig, ax = plt.subplots(
    figsize=(7.2, 6.2)
)


ax.scatter(
    y_c_test,
    y_pred_xgb_c,
    s=30,
    alpha=0.65,
    color=XGB_COLOR,
    edgecolors="black",
    linewidths=0.25,
    marker="o"
)


mn = min(
    y_c_test.min(),
    y_pred_xgb_c.min()
)

mx = max(
    y_c_test.max(),
    y_pred_xgb_c.max()
)


ax.plot(
    [mn, mx],
    [mn, mx],
    linestyle="--",
    color="black",
    linewidth=1.2
)


ax.set_xlabel(
    "Actual MPG"
)

ax.set_ylabel(
    "Predicted MPG"
)

ax.set_title(
    "Actual versus predicted MPG — Configuration model",
    fontsize=13,
    fontweight="bold",
    pad=12
)


remove_spines(ax)


plt.tight_layout()


save_tiff(
    fig,
    "Figure_2_Actual_vs_Predicted_Config.tiff"
)


# ============================================================
# ============================================================
# FIGURE 3
# ACTUAL VS PREDICTED — BRAND MODEL
# ============================================================
#
# Paper:
# "Actual vs. Predicted avg_mpg for the Brand Model
#  (weighted ensemble, held-out test set)."
# ============================================================


fig, ax = plt.subplots(
    figsize=(7.2, 6.2)
)


ax.scatter(
    y_b_test,
    y_pred_ens_b,
    s=30,
    alpha=0.60,
    color=ENS_COLOR,
    edgecolors="black",
    linewidths=0.25,
    marker="o"
)


mn = min(
    y_b_test.min(),
    y_pred_ens_b.min()
)

mx = max(
    y_b_test.max(),
    y_pred_ens_b.max()
)


ax.plot(
    [mn, mx],
    [mn, mx],
    linestyle="--",
    color="black",
    linewidth=1.2
)


ax.set_xlabel(
    "Actual avg_mpg"
)

ax.set_ylabel(
    "Predicted avg_mpg"
)

ax.set_title(
    "Actual versus predicted avg_mpg — Brand model",
    fontsize=13,
    fontweight="bold",
    pad=12
)


remove_spines(ax)


plt.tight_layout()


save_tiff(
    fig,
    "Figure_3_Actual_vs_Predicted_Brand.tiff"
)


# ============================================================
# ============================================================
# SHAP EXPLAINERS
# ============================================================
#
# Paper explicitly states that for the Brand path,
# SHAP global explanations are calculated for the
# XGBoost sub-model because it contributes 60% to
# the ensemble.
# ============================================================


explainer_c = shap.TreeExplainer(
    model_xgb_c
)

shap_vals_c = explainer_c.shap_values(
    X_test_c.values
)


explainer_b = shap.TreeExplainer(
    model_xgb_b
)

shap_vals_b = explainer_b.shap_values(
    X_test_b.values
)


# ============================================================
# ============================================================
# FIGURE 6
# SHAP SUMMARY — CONFIGURATION
# ============================================================
#
# Two panels:
# (a) SHAP feature importance
# (b) SHAP feature impact / beeswarm
#
# Short x-axis labels prevent the previous overlap problem.
# ============================================================


fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5.5)
)


# ------------------------------------------------------------
# Figure 6(a)
# ------------------------------------------------------------

plt.sca(
    axes[0]
)

shap.summary_plot(
    shap_vals_c,
    X_test_c,
    feature_names=config_features,
    plot_type="bar",
    max_display=len(config_features),
    show=False,
    plot_size=None
)


axes[0].set_title(
    "SHAP feature importance",
    fontsize=13,
    fontweight="bold",
    pad=12
)

axes[0].set_xlabel(
    "Mean |SHAP value|",
    fontsize=11,
    labelpad=8
)

remove_spines(
    axes[0]
)


# ------------------------------------------------------------
# Figure 6(b)
# ------------------------------------------------------------

plt.sca(
    axes[1]
)

shap.summary_plot(
    shap_vals_c,
    X_test_c,
    feature_names=config_features,
    max_display=len(config_features),
    show=False,
    plot_size=None,
    color_bar=True
)


axes[1].set_title(
    "SHAP feature impact",
    fontsize=13,
    fontweight="bold",
    pad=12
)

axes[1].set_xlabel(
    "SHAP value",
    fontsize=11,
    labelpad=8
)

remove_spines(
    axes[1]
)


# ------------------------------------------------------------
# Main title
# ------------------------------------------------------------

fig.suptitle(
    "SHAP analysis for the Configuration model",
    fontsize=14,
    fontweight="bold",
    y=0.98
)


# Panel labels

fig.text(
    0.25,
    0.015,
    "(a)",
    ha="center",
    fontsize=11
)

fig.text(
    0.75,
    0.015,
    "(b)",
    ha="center",
    fontsize=11
)


# Spacing
plt.subplots_adjust(
    left=0.08,
    right=0.93,
    bottom=0.17,
    top=0.84,
    wspace=0.48
)


save_tiff(
    fig,
    "Figure_6_SHAP_Summary_Config.tiff"
)


# ============================================================
# ============================================================
# FIGURE 7
# SHAP SUMMARY — BRAND
# ============================================================


fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5.5)
)


# ------------------------------------------------------------
# Figure 7(a)
# ------------------------------------------------------------

plt.sca(
    axes[0]
)

shap.summary_plot(
    shap_vals_b,
    X_test_b,
    feature_names=brand_features,
    plot_type="bar",
    max_display=len(brand_features),
    show=False,
    plot_size=None
)


axes[0].set_title(
    "SHAP feature importance",
    fontsize=13,
    fontweight="bold",
    pad=12
)

axes[0].set_xlabel(
    "Mean |SHAP value|",
    fontsize=11,
    labelpad=8
)

remove_spines(
    axes[0]
)


# ------------------------------------------------------------
# Figure 7(b)
# ------------------------------------------------------------

plt.sca(
    axes[1]
)

shap.summary_plot(
    shap_vals_b,
    X_test_b,
    feature_names=brand_features,
    max_display=len(brand_features),
    show=False,
    plot_size=None,
    color_bar=True
)


axes[1].set_title(
    "SHAP feature impact",
    fontsize=13,
    fontweight="bold",
    pad=12
)

axes[1].set_xlabel(
    "SHAP value",
    fontsize=11,
    labelpad=8
)

remove_spines(
    axes[1]
)


fig.suptitle(
    "SHAP analysis for the Brand model",
    fontsize=14,
    fontweight="bold",
    y=0.98
)


fig.text(
    0.25,
    0.015,
    "(a)",
    ha="center",
    fontsize=11
)

fig.text(
    0.75,
    0.015,
    "(b)",
    ha="center",
    fontsize=11
)


plt.subplots_adjust(
    left=0.08,
    right=0.93,
    bottom=0.17,
    top=0.84,
    wspace=0.48
)


save_tiff(
    fig,
    "Figure_7_SHAP_Summary_Brand.tiff"
)


# ============================================================
# ============================================================
# FIGURE 8
# LIME LOCAL EXPLANATION — CONFIGURATION
# ============================================================
#
# This is a LOCAL explanation for Instance 0.
# Green = contribution toward higher predicted MPG
# Red   = contribution toward lower predicted MPG
# ============================================================


lime_exp_c = lime.lime_tabular.LimeTabularExplainer(
    X_train_c.values,
    feature_names=config_features,
    mode="regression",
    random_state=42
)


def ensemble_pred_c(x):

    return (
        XGB_W * model_xgb_c.predict(x)
        +
        RF_W * model_rf_c.predict(x)
    )


lime_instance_c = X_test_c.values[0]


exp_c = lime_exp_c.explain_instance(
    lime_instance_c,
    ensemble_pred_c,
    num_features=len(config_features),
    num_samples=5000
)


lime_c = exp_c.as_list()


lime_features_c = [
    item[0]
    for item in lime_c
]

lime_weights_c = np.array([
    item[1]
    for item in lime_c
])


fig, ax = plt.subplots(
    figsize=(8.5, 5.5)
)


colors_c = [
    POSITIVE_COLOR if w > 0
    else NEGATIVE_COLOR
    for w in lime_weights_c
]


hatches_c = [
    POSITIVE_HATCH if w > 0
    else NEGATIVE_HATCH
    for w in lime_weights_c
]


bars = ax.barh(
    lime_features_c,
    lime_weights_c,
    color=colors_c,
    edgecolor="black",
    linewidth=0.5
)


for bar, hatch in zip(
    bars,
    hatches_c
):

    bar.set_hatch(
        hatch
    )


ax.axvline(
    0,
    color="black",
    linewidth=1
)


ax.set_xlabel(
    "LIME weight"
)

ax.set_title(
    "LIME local explanation — Configuration model",
    fontsize=13,
    fontweight="bold",
    pad=12
)


remove_spines(ax)


plt.tight_layout()


save_tiff(
    fig,
    "Figure_8_LIME_Local_Config.tiff"
)


# ============================================================
# ============================================================
# FIGURE 9
# LIME LOCAL EXPLANATION — BRAND
# ============================================================


lime_exp_b = lime.lime_tabular.LimeTabularExplainer(
    X_train_b.values,
    feature_names=brand_features,
    mode="regression",
    random_state=42
)


def ensemble_pred_b(x):

    return (
        XGB_W * model_xgb_b.predict(x)
        +
        RF_W * model_rf_b.predict(x)
    )


lime_instance_b = X_test_b.values[0]


exp_b = lime_exp_b.explain_instance(
    lime_instance_b,
    ensemble_pred_b,
    num_features=len(brand_features),
    num_samples=5000
)


lime_b = exp_b.as_list()


lime_features_b = [
    item[0]
    for item in lime_b
]

lime_weights_b = np.array([
    item[1]
    for item in lime_b
])


fig, ax = plt.subplots(
    figsize=(8.5, 5.5)
)


colors_b = [
    POSITIVE_COLOR if w > 0
    else NEGATIVE_COLOR
    for w in lime_weights_b
]


hatches_b = [
    POSITIVE_HATCH if w > 0
    else NEGATIVE_HATCH
    for w in lime_weights_b
]


bars = ax.barh(
    lime_features_b,
    lime_weights_b,
    color=colors_b,
    edgecolor="black",
    linewidth=0.5
)


for bar, hatch in zip(
    bars,
    hatches_b
):

    bar.set_hatch(
        hatch
    )


ax.axvline(
    0,
    color="black",
    linewidth=1
)


ax.set_xlabel(
    "LIME weight"
)

ax.set_title(
    "LIME local explanation — Brand model",
    fontsize=13,
    fontweight="bold",
    pad=12
)


remove_spines(ax)


plt.tight_layout()


save_tiff(
    fig,
    "Figure_9_LIME_Local_Brand.tiff"
)


# ============================================================
# ============================================================
# 13. AGGREGATED LIME IMPORTANCE
# ============================================================
#
# IMPORTANT:
#
# Figures 10 and 11 are NOT single-instance comparisons.
#
# The paper says:
#
#   SHAP = mean absolute SHAP value
#   LIME = aggregated local importance
#
# Therefore we calculate LIME importance over multiple
# test instances and then normalize it.
#
# We use feature indices from LIME's local_exp dictionary,
# rather than trying to parse text conditions.
# ============================================================


def aggregate_lime_importance(
    model_xgb,
    model_rf,
    X_train,
    X_test,
    feature_names,
    max_instances=None,
    random_state=42
):

    explainer = lime.lime_tabular.LimeTabularExplainer(
        X_train.values,
        feature_names=feature_names,
        mode="regression",
        random_state=random_state
    )


    def ensemble_predict(x):

        return (
            XGB_W * model_xgb.predict(x)
            +
            RF_W * model_rf.predict(x)
        )


    n_instances = len(X_test)


    if max_instances is not None:

        n_instances = min(
            n_instances,
            max_instances
        )


    # deterministic evenly distributed instances
    indices = np.linspace(
        0,
        len(X_test) - 1,
        n_instances,
        dtype=int
    )


    accumulated = np.zeros(
        len(feature_names),
        dtype=float
    )


    counts = np.zeros(
        len(feature_names),
        dtype=float
    )


    print(
        f"\n🔄 Aggregating LIME over "
        f"{len(indices)} instances..."
    )


    for counter, idx in enumerate(
        indices,
        start=1
    ):

        explanation = explainer.explain_instance(
            X_test.values[idx],
            ensemble_predict,
            num_features=len(feature_names),
            num_samples=1000
        )


        # LIME regression label is 1
        local_exp = explanation.local_exp.get(
            1,
            []
        )


        for feature_idx, weight in local_exp:

            if 0 <= feature_idx < len(
                feature_names
            ):

                accumulated[
                    feature_idx
                ] += abs(weight)

                counts[
                    feature_idx
                ] += 1


        if (
            counter % 25 == 0
            or
            counter == len(indices)
        ):

            print(
                f"   Processed "
                f"{counter}/{len(indices)}"
            )


    # Mean absolute local importance
    counts[
        counts == 0
    ] = 1


    mean_importance = (
        accumulated
        /
        counts
    )


    return mean_importance


# ============================================================
# NUMBER OF INSTANCES
# ============================================================
#
# Configuration test set is small (~80), so use ALL.
#
# Brand test set can be much larger.
# Use all test instances as well so the comparison is
# consistent with the paper's aggregated LIME approach.
#
# If your Brand dataset is very large, this can take time.
# ============================================================


LIME_CONFIG_INSTANCES = len(
    X_test_c
)

LIME_BRAND_INSTANCES = len(
    X_test_b
)


# ============================================================
# CONFIGURATION — AGGREGATED LIME
# ============================================================

lime_global_c = aggregate_lime_importance(
    model_xgb_c,
    model_rf_c,
    X_train_c,
    X_test_c,
    config_features,
    max_instances=LIME_CONFIG_INSTANCES,
    random_state=42
)


# ============================================================
# BRAND — AGGREGATED LIME
# ============================================================

lime_global_b = aggregate_lime_importance(
    model_xgb_b,
    model_rf_b,
    X_train_b,
    X_test_b,
    brand_features,
    max_instances=LIME_BRAND_INSTANCES,
    random_state=42
)


# ============================================================
# 14. GLOBAL SHAP IMPORTANCE
# ============================================================


shap_global_c = np.mean(
    np.abs(shap_vals_c),
    axis=0
)


shap_global_b = np.mean(
    np.abs(shap_vals_b),
    axis=0
)


# ============================================================
# 15. NORMALIZE BOTH METHODS
# ============================================================


def normalize(values):

    values = np.asarray(
        values,
        dtype=float
    )

    total = np.sum(values)

    if total == 0:

        return np.zeros_like(
            values
        )

    return values / total


shap_norm_c = normalize(
    shap_global_c
)

lime_norm_c = normalize(
    lime_global_c
)


shap_norm_b = normalize(
    shap_global_b
)

lime_norm_b = normalize(
    lime_global_b
)


# ============================================================
# 16. SPEARMAN CORRELATION
# ============================================================


rho_c, p_c = spearmanr(
    shap_norm_c,
    lime_norm_c
)


rho_b, p_b = spearmanr(
    shap_norm_b,
    lime_norm_b
)


print("\n" + "=" * 60)

print(
    f"Configuration SHAP vs LIME:"
)

print(
    f"Spearman rho = {rho_c:.3f}"
)

print(
    f"p-value      = {p_c:.4f}"
)


print(
    f"\nBrand SHAP vs LIME:"
)

print(
    f"Spearman rho = {rho_b:.3f}"
)

print(
    f"p-value      = {p_b:.4f}"
)

print("=" * 60)


# ============================================================
# ============================================================
# FIGURE 10
# SHAP VS LIME — CONFIGURATION
# ============================================================
#
# SHAP = normalized mean |SHAP value|
# LIME = normalized aggregated local importance
# ============================================================


x = np.arange(
    len(config_features)
)

width = 0.36


fig, ax = plt.subplots(
    figsize=(10, 5.8)
)


bars1 = ax.bar(
    x - width / 2,
    shap_norm_c,
    width,
    label="SHAP",
    color=XGB_COLOR,
    edgecolor="black",
    linewidth=0.5,
    hatch="///"
)


bars2 = ax.bar(
    x + width / 2,
    lime_norm_c,
    width,
    label="LIME",
    color=ENS_COLOR,
    edgecolor="black",
    linewidth=0.5,
    hatch="\\\\\\"
)


ax.set_xticks(
    x
)

ax.set_xticklabels(
    config_features,
    rotation=35,
    ha="right",
    fontsize=9
)


ax.set_ylabel(
    "Normalized importance"
)


ax.set_title(
    "SHAP versus LIME — Configuration model",
    fontsize=13,
    fontweight="bold",
    pad=12
)


legend = ax.legend(
    frameon=False,
    loc="upper right"
)

clean_legend(
    legend
)


remove_spines(ax)


plt.tight_layout()


save_tiff(
    fig,
    "Figure_10_SHAP_vs_LIME_Config.tiff"
)


# ============================================================
# ============================================================
# FIGURE 11
# SHAP VS LIME — BRAND
# ============================================================


x = np.arange(
    len(brand_features)
)


fig, ax = plt.subplots(
    figsize=(10, 5.8)
)


bars1 = ax.bar(
    x - width / 2,
    shap_norm_b,
    width,
    label="SHAP",
    color=XGB_COLOR,
    edgecolor="black",
    linewidth=0.5,
    hatch="///"
)


bars2 = ax.bar(
    x + width / 2,
    lime_norm_b,
    width,
    label="LIME",
    color=ENS_COLOR,
    edgecolor="black",
    linewidth=0.5,
    hatch="\\\\\\"
)


ax.set_xticks(
    x
)

ax.set_xticklabels(
    brand_features,
    rotation=35,
    ha="right",
    fontsize=9
)


ax.set_ylabel(
    "Normalized importance"
)


ax.set_title(
    "SHAP versus LIME — Brand model",
    fontsize=13,
    fontweight="bold",
    pad=12
)


legend = ax.legend(
    frameon=False,
    loc="upper right"
)

clean_legend(
    legend
)


remove_spines(ax)


plt.tight_layout()


save_tiff(
    fig,
    "Figure_11_SHAP_vs_LIME_Brand.tiff"
)


# ============================================================
# ============================================================
# 17. FINAL FILE CHECK
# ============================================================


expected_files = [

    "Figure_2_Actual_vs_Predicted_Config.tiff",

    "Figure_3_Actual_vs_Predicted_Brand.tiff",

    "Figure_6_SHAP_Summary_Config.tiff",

    "Figure_7_SHAP_Summary_Brand.tiff",

    "Figure_8_LIME_Local_Config.tiff",

    "Figure_9_LIME_Local_Brand.tiff",

    "Figure_10_SHAP_vs_LIME_Config.tiff",

    "Figure_11_SHAP_vs_LIME_Brand.tiff"
]


print("\n\n" + "=" * 70)

print(
    "FINAL JOURNAL FIGURE CHECK"
)

print("=" * 70)


for filename in expected_files:

    filepath = os.path.join(
        OUTPUT_DIR,
        filename
    )

    if os.path.exists(filepath):

        size_mb = (
            os.path.getsize(filepath)
            /
            (1024 * 1024)
        )

        print(
            f"✅ {filename} "
            f"({size_mb:.2f} MB)"
        )

    else:

        print(
            f"❌ MISSING: {filename}"
        )


# ============================================================
# 18. CREATE ZIP
# ============================================================


ZIP_NAME = (
    "Paper_Final_Figures_"
    "2_3_6_7_8_9_10_11_"
    "600dpi.zip"
)


if os.path.exists(
    ZIP_NAME
):

    os.remove(
        ZIP_NAME
    )


with zipfile.ZipFile(
    ZIP_NAME,
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    for filename in expected_files:

        filepath = os.path.join(
            OUTPUT_DIR,
            filename
        )

        if os.path.exists(
            filepath
        ):

            zipf.write(
                filepath,
                arcname=filename
            )


# ============================================================
# 19. FINAL SUMMARY
# ============================================================


print("\n" + "=" * 70)

print(
    "🎉 FINAL FIGURES READY"
)

print("=" * 70)

print(
    "\nNumber of figures:",
    len(expected_files)
)

print(
    "\nFormat: COLOUR TIFF"
)

print(
    "Resolution: 600 DPI"
)

print(
    "Colour requirement >=300 DPI: ✅"
)

print(
    "High-resolution print quality: ✅"
)

print(
    "Unnecessary boundary lines removed: ✅"
)

print(
    "Only required paper figures: ✅"
)

print(
    "\nZIP:"
)

print(
    ZIP_NAME
)

print(
    "\nOutput folder:"
)

print(
    OUTPUT_DIR
)

print("=" * 70)


# ============================================================
# 20. DOWNLOAD ZIP AUTOMATICALLY
# ============================================================

from google.colab import files

files.download(
    ZIP_NAME
)

✅ Brand source dataset found: car_specs_final.csv
✅ Brand actual target loaded: 2361 test samples
✅ Saved: Figure_2_Actual_vs_Predicted_Config.tiff
✅ Saved: Figure_3_Actual_vs_Predicted_Brand.tiff
✅ Saved: Figure_6_SHAP_Summary_Config.tiff
✅ Saved: Figure_7_SHAP_Summary_Brand.tiff
✅ Saved: Figure_8_LIME_Local_Config.tiff
✅ Saved: Figure_9_LIME_Local_Brand.tiff

🔄 Aggregating LIME over 80 instances...
   Processed 25/80
   Processed 50/80
   Processed 75/80
   Processed 80/80

🔄 Aggregating LIME over 2361 instances...
   Processed 25/2361
   Processed 50/2361
   Processed 75/2361
   Processed 100/2361
   Processed 125/2361
   Processed 150/2361
   Processed 175/2361
   Processed 200/2361
   Processed 225/2361
   Processed 250/2361
   Processed 275/2361
   Processed 300/2361
   Processed 325/2361
   Processed 350/2361
   Processed 375/2361
   Processed 400/2361
   Processed 425/2361
   Processed 450/2361
   Processed 475/2361
   Processed 500/2361
   Processed 525/2361
   Processed 550/2

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>